# Системы поддержки принятия решений

**Тема: «Дополнительное задание. Оптимизация структуры энергосистемы»**

Исходная постановка — § 3 «Применение линейного программирования к энергетике», с. 60–63 предоставленного учебного пособия. Минимизация эксплуатационных расходов заменена максимизацией отношения годовой выработки энергии к затратам на строительство. Дробная цель преобразуется методом Чарнса–Купера, затем отдельно решаются прямая и двойственная задачи ЛП.

Работа содержит исходный текст, учебный численный пример, вывод моделей, расчёт в Pyomo и проверку сильной двойственности. Численные параметры сформированы для этой работы и не являются данными конкретного энергетического проекта.

In [1]:
import pyomo.environ as pyo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.float_format", "{:,.6f}".format)
pd.set_option("display.max_columns", 12)
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 120,
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
})

solver = None
for solver_name in ("appsi_highs", "highs", "cbc"):
    candidate = pyo.SolverFactory(solver_name)
    if candidate.available(exception_flag=False):
        solver = candidate
        break

if solver is None:
    raise RuntimeError("Решатель не найден. Установите highspy: python -m pip install highspy")

def solve_model(model):
    result = solver.solve(model, tee=False)
    pyo.assert_optimal_termination(result)
    return result

print(f"Решатель: {solver_name}")

Решатель: appsi_highs


## Условие

**§ 3. ПРИМЕНЕНИЕ ЛИНЕЙНОГО ПРОГРАММИРОВАНИЯ К ЭНЕРГЕТИКЕ**

В настоящее время разработано и освоено значительное число проектов электростанций, использующих различные источники энергии. Имеется ряд стандартных проектов гидроэлектростанций и тепловых станций, применяющих различные виды горючего, — уголь, нефть, торф, горючие сланцы и др. Освоено несколько типов атомных электростанций. Разработаны и приливные электростанции, использующие энергию морского прилива. Рассматриваются и другие принципы получения электрической энергии (например, преобразование солнечной энергии, энергии ветра, энергии вулканической деятельности земли и т. д.).

При решении вопроса об электрификации района, располагающего различными источниками энергии, важно выбрать рациональный набор типов станций, способных удовлетворить потребности района при минимальных затратах. Любой набор станций, составляющих энергетическую систему района, должен (см. [28]) обеспечить заданные значения гарантированной мощности ($A$), пиковой мощности ($B$) и заданный уровень годовой выработки энергии ($C$). При этом затраты на капитальное строительство не должны превысить фиксированной суммы ($D$). Величины $A$, $B$, $C$ и $D$ определяются перспективным планом развития экономики района.

Пусть природные условия района позволяют рассматривать возможность строительства в районе любого из $n$ типов электростанций. Обозначим через $x_j$ число станций $j$-го типа. Введем, кроме того, следующие обозначения:

$a_j$ — гарантированная мощность, обеспечиваемая одной станцией $j$-го типа;

$b_j$ — пиковая мощность одной станции $j$-го типа,

$c_j$ — годовая выработка энергии одной станцией $j$-го типа;

$d_j$ — затраты на строительство одной станции $j$-го типа;

$f_j$ — суммарные годовые эксплуатационные расходы (обслуживание и управление) на одну станцию $j$-го типа.

Оптимальная в смысле минимума эксплуатационных расходов энергосистема состоит из $x_j$ станций $j$-го типа ($j=1,2,\ldots,n$). Числа $x_j$ обеспечивают минимум линейной формы

$$
L=\sum_{j=1}^{n}f_jx_j
\tag{3.1}
$$

при условиях:

$$
\sum_{j=1}^{n}a_jx_j\ge A,
\tag{3.2}
$$

$$
\sum_{j=1}^{n}b_jx_j\ge B,
\tag{3.3}
$$

$$
\sum_{j=1}^{n}c_jx_j\ge C,
\tag{3.4}
$$

$$
\sum_{j=1}^{n}d_jx_j\le D,
\tag{3.5}
$$

$$
x_j\ge0,\quad j=1,2,\ldots,n.
\tag{3.6}
$$

Показатель качества (3.1) решения задачи равен суммарным годовым расходам всех станций энергосистемы на обслуживание, управление и уход за оборудованием. Ограничения (3.2)—(3.4) обеспечивают заданный уровень гарантированной и пиковой мощностей системы и требуемую годовую выработку энергии по району. Ограничение (3.5) гарантирует, что суммарные затраты на строительство станций энергосистемы не превысят размеров запланированных капиталовложений. Естественные ограничения (3.6) определяются физическим смыслом переменных $x_j$. Заметим, кроме того что по своему физическому смыслу переменные $x_j$ — целые числа. Решение задачи (3.1)—(3.6) может привести к дробным значениям компонент $x_j$. Округление значений $x_j$ до ближайших целых чисел, как правило, не приводит к существенным отклонениям от оптимального плана, если $x_j\gg1$. При малых значениях компонент оптимального плана задачи (3.1)—(3.6), округление составляющих $x_j$ решения недопустимо. Полученный таким образом план может оказаться существенно менее эффективным, чем оптимальный целочисленный план задачи. В этом случае следует привлечь для решения задачи методы так называемого целочисленного программирования.

Задача (3.1)—(3.6) содержит 4 условия общего вида ($m=4$). Отсюда следует (это будет обосновано в § 4 главы 5), что оптимальная энергосистема, определяемая решением задачи (3.1)—(3.6), будет содержать не более четырех типов станций. Необходимость учета дополнительных ограничений заставляет, вообще говоря, увеличить число типов станций, входящих в состав оптимальной энергосистемы. Дополнительные условия могут определяться, например, ограниченным количеством того или иного оборудования, необходимого всем или некоторым станциям системы. Лимитирующим фактором при составлении плана развития энергетики района может оказаться также ограниченная возможность обеспечения района квалифицированными инженерно-техническими кадрами. Все подобные ограничения записываются так же, как и ограничение (3.5).

Строительство станций по новым проектам требует создания нового оборудования. Развертывание нового производства экономически оправдано лишь в тех случаях, когда потребность в его продукции превышает некоторый минимум. Отсюда новые ограничения вида

$$
\sum_{j=1}^{n}e_j^{(i)}x_j\ge\varepsilon_{(i)},\quad i=1,2,\ldots,s,
$$

где $e_j^{(i)}$ — потребность каждой станции $j$-го типа в $i$-м виде нового оборудования, а

$\varepsilon_{(i)}$ — минимальный объем продукции $i$-го вида, оправдывающий развертывание нового производства.

Отдельные типы станций, нерентабельные с точки зрения современных технических возможностей, являются, тем не менее, перспективными предприятиями, и накопление опыта их строительства и эксплуатации — необходимое условие технического прогресса. С другой стороны, имеются определенные проекты станций, экономичных в настоящее время, но по тем или иным причинам неперспективных (например, из-за ограниченных ресурсов соответствующего источника энергии). Учет последних замечаний приводит к замене ограничений вида (3.6) условиями вида

$$
\alpha_j\le x_j\le\beta_j,\quad j=1,2,\ldots,n,
$$

где $\alpha_j$ и $\beta_j$ предполагаются заданными величинами. Значения $\alpha_j$ и $\beta_j$ определяются при планировании на более высоком уровне, исходящем из более далекой перспективы.

Известны и другие энергетические модели, анализ которых сводится к решению задач линейного и нелинейного программирования. Некоторые из них будут рассмотрены в последующих главах при изложении специальных линейных условных экстремальных задач.


**Уточнение задания.** В оригинальной задаче (3.1) минимизируются суммарные годовые эксплуатационные расходы. В этой работе требуется сохранить ограничения (3.2)–(3.6), заменить цель на максимизацию $\sum_j c_jx_j/\sum_j d_jx_j$ при положительном знаменателе, выполнить преобразование Чарнса–Купера и построить двойственную задачу к полученной ЛП.

## Дано

В предоставленном параграфе численные значения коэффициентов отсутствуют. Ниже задан **воспроизводимый учебный пример для пяти типов станций**. Все числа в таблицах этого раздела — параметры примера; обозначения и смысл величин сохранены из источника.

- $a_j$ — гарантированная доступная мощность станции, МВт.
- $b_j$ — пиковая мощность станции, МВт.
- $c_j$ — годовая выработка, ГВт·ч/год.
- $d_j$ — затраты на строительство, млн долл.
- $f_j$ — годовые эксплуатационные расходы на обслуживание и управление, млн долл./год.

| Тип станции | $a_j$, МВт | $b_j$, МВт | $c_j$, ГВт·ч/год | $d_j$, млн долл. | $f_j$, млн долл./год |
|:---|---:|---:|---:|---:|---:|
| ГЭС | 50 | 100 | 400 | 300 | 6 |
| ТЭЦ | 160 | 200 | 1000 | 400 | 10 |
| АЭС | 900 | 1000 | 7500 | 6000 | 180 |
| ВЭС | 0 | 100 | 350 | 120 | 4,5 |
| СЭС | 0 | 100 | 180 | 70 | 1,5 |

Требования района и бюджет:

$$A=320\ \text{МВт},\quad B=900\ \text{МВт},\quad
C=3200\ \text{ГВт·ч/год},\quad D=1300\ \text{млн долл.}$$

Гарантированная мощность означает доступную мощность при расчётных условиях; она не предполагает обязательной непрерывной выдачи этой мощности весь год. Для ветровой и солнечной станций без накопителей в данном примере консервативно принято $a_j=0$. Под одной ВЭС или СЭС понимается станция как комплекс установок заданной мощности.

Для формирования таблицы в коде используются пиковые мощности и следующие удельные параметры:

| Тип | Доля гарантированной мощности | Часы использования пиковой мощности за год | Строительство, долл./кВт | Обслуживание, долл./(кВт·год) |
|:---|---:|---:|---:|---:|
| ГЭС | 0,5 | 4000 | 3000 | 60 |
| ТЭЦ | 0,8 | 5000 | 2000 | 50 |
| АЭС | 0,9 | 7500 | 6000 | 180 |
| ВЭС | 0 | 3500 | 1200 | 45 |
| СЭС | 0 | 1800 | 700 | 15 |

Используются соотношения $a_j=r_jb_j$, $c_j=b_jh_j/1000$, $d_j=b_jk_j/1000$, $f_j=b_jo_j/1000$, где $r_j,h_j,k_j,o_j$ — соответствующие столбцы второй таблицы. Данные неслучайные, поэтому при повторном запуске воспроизводится тот же пример. Проверяются условия $0\le a_j\le b_j$, $0<c_j\le8{,}76b_j$ и $d_j>0$.

Для сопоставления порядка величин использована система показателей капитальных затрат, расходов на обслуживание и коэффициента использования мощности из [Annual Technology Baseline](https://atb.nrel.gov/electricity/2024b/index). Числа примера выбраны для учебного расчёта и не являются цитатой из этой базы.

Численная модель использует базовые ограничения (3.2)–(3.6). Упомянутые в конце параграфа дополнительные ограничения на оборудование и границы $\alpha_j,\beta_j$ не вводятся: параметры для них в задании не заданы.

## Математическая модель

**Переменные и исходная модель (3.1)–(3.6).** Пусть $x_j\ge0$ — количество станций типа $j$. Исходная задача имеет вид

$$
\begin{aligned}
\min\quad &L=\sum_{j=1}^{n}f_jx_j,\\
\text{при}\quad &\sum_j a_jx_j\ge A,\qquad \sum_j b_jx_j\ge B,\\
&\sum_j c_jx_j\ge C,\qquad \sum_j d_jx_j\le D,\\
&x_j\in\mathbb R_{\ge0},\qquad j=1,\ldots,n.
\end{aligned}
$$

Рассматривается непрерывная постановка ЛП, как в формулах источника. Для неё возможны дробные эквивалентные объёмы станций. Физическое требование целочисленности оговорено в исходном тексте; в этом примере найденный оптимальный план оказывается целым без округления.

**Модифицированная дробно-линейная модель.** Обозначим годовую выработку $E(x)=\sum_jc_jx_j$, капитальные затраты $K(x)=\sum_jd_jx_j$. Новая цель:

$$
\max\quad \eta(x)=\frac{E(x)}{K(x)}
=\frac{\sum_jc_jx_j}{\sum_jd_jx_j}
$$

при тех же ограничениях и $K(x)>0$. Единица эффективности — **ГВт·ч/год на 1 млн долл. строительства**. Коэффициенты $f_j$ сохраняются в данных и используются для справочного расчёта эксплуатационных расходов; новая цель от них не зависит.

Поскольку $C>0$, любой допустимый план ненулевой, а при $d_j>0$ это обеспечивает $K(x)>0$. При этом $K(x)\le D$. Эффективность ограничена сверху: она является взвешенным средним величин $c_j/d_j$ с весами $d_jx_j/K(x)$, поэтому $\eta(x)\le\max_j(c_j/d_j)$.

**Модель 1. Прямая ЛП после замены Чарнса–Купера.** Введём

$$t=\frac1{K(x)},\qquad y_j=tx_j,\qquad x_j=\frac{y_j}{t}.$$

Нормировка знаменателя даёт $\sum_jd_jy_j=1$. Для удобства построения двойственной задачи запишем все неравенства в форме «не больше нуля»:

$$
\begin{aligned}
\max\quad &P=\sum_jc_jy_j,\\
\text{при}\quad &At-\sum_ja_jy_j\le0, &&(u)\\
&Bt-\sum_jb_jy_j\le0, &&(v)\\
&Ct-\sum_jc_jy_j\le0, &&(w)\\
&\sum_jd_jy_j-Dt\le0, &&(z)\\
&\sum_jd_jy_j=1, &&(\lambda)\\
&y_j\ge0,\quad t\ge0.
\end{aligned}
$$

Из бюджетного ограничения и нормировки следует $1\le Dt$, то есть $t\ge1/D>0$. Поэтому не требуется задавать решателю строгое неравенство или искусственный малый порог. Восстановление $x_j=y_j/t$ корректно, а $P=\eta(x)$.

**Модель 2. Двойственная ЛП.** Неравенствам прямой задачи соответствуют $u,v,w,z\ge0$, равенству нормировки — свободная переменная $\lambda\in\mathbb R$. Для каждой неотрицательной переменной $y_j$ и отдельно для $t$ получаем:

$$
\begin{aligned}
\min\quad &Q=\lambda,\\
\text{при}\quad &(\lambda+z)d_j-u a_j-v b_j-w c_j\ge c_j,
&&j=1,\ldots,n,\\
&Au+Bv+Cw-Dz\ge0,\\
&u,v,w,z\ge0,\qquad\lambda\in\mathbb R.
\end{aligned}
$$

Цель равна $\lambda$: правые части четырёх неравенств прямой ЛП равны нулю, а правая часть нормировки — единице. Последнее двойственное ограничение соответствует переменной $t$ с нулевым коэффициентом в прямой цели.

Слабая двойственность даёт $P\le Q$. Для допустимых оптимальных решений выполняется сильная двойственность $P^*=Q^*$. Дополняющая нежёсткость требует нулевых произведений двойственных переменных на запасы соответствующих ограничений; для каждого типа станции также выполняется

$$y_j\bigl[(\lambda+z)d_j-u a_j-v b_j-w c_j-c_j\bigr]=0,$$

а для масштаба — $t(Au+Bv+Cw-Dz)=0$.

**Смысл двойственных оценок.** Значения $u,v,w,z$ относятся к записанной преобразованной ЛП. Для малых изменений исходных требований и бюджета, пока сохраняется оптимальная структура, предельные изменения эффективности равны

$$
\frac{\partial\eta^*}{\partial A}=-t^*u^*,\quad
\frac{\partial\eta^*}{\partial B}=-t^*v^*,\quad
\frac{\partial\eta^*}{\partial C}=-t^*w^*,\quad
\frac{\partial\eta^*}{\partial D}=t^*z^*.
$$

В выводе отдельно представлены двойственные переменные и эти предельные эффекты, чтобы сохранить различие между правыми частями преобразованной ЛП и исходными параметрами района.

## Найти

Оптимальное количество станций каждого типа $x_j$, обеспечивающее максимальное отношение годовой выработки энергии к затратам на строительство при заданных требованиях района и бюджете; значения целевых функций прямой и двойственной задач и подтверждение их равенства.

## Решение

**Формирование учебных данных.** Параметры создаются из мощностей, часов использования и удельных затрат, приведённых в разделе «Дано».

In [2]:
station_specs = pd.DataFrame({
    "name": ["ГЭС", "ТЭЦ", "АЭС", "ВЭС", "СЭС"],
    "peak_mw": [100, 200, 1000, 100, 100],
    "firm_share": [0.5, 0.8, 0.9, 0, 0],
    "full_load_hours": [4000, 5000, 7500, 3500, 1800],
    "capex_per_kw": [3000, 2000, 6000, 1200, 700],
    "annual_om_per_kw": [60, 50, 180, 45, 15],
}, index=["hydro", "chp", "nuclear", "wind", "solar"])

station_data = pd.DataFrame(index=station_specs.index)
station_data["a"] = station_specs["peak_mw"] * station_specs["firm_share"]
station_data["b"] = station_specs["peak_mw"]
station_data["c"] = station_specs["peak_mw"] * station_specs["full_load_hours"] / 1000
station_data["d"] = station_specs["peak_mw"] * station_specs["capex_per_kw"] / 1000
station_data["f"] = station_specs["peak_mw"] * station_specs["annual_om_per_kw"] / 1000

required_firm = 320.0
required_peak = 900.0
required_energy = 3200.0
construction_budget = 1300.0

assert (station_data["a"] >= 0).all()
assert (station_data["a"] <= station_data["b"]).all()
assert (station_data["c"] > 0).all()
assert (station_data["c"] <= 8.76 * station_data["b"]).all()
assert (station_data["d"] > 0).all()

input_table = station_data.rename(columns={
    "a": "a, МВт",
    "b": "b, МВт",
    "c": "c, ГВт·ч/год",
    "d": "d, млн долл.",
    "f": "f, млн долл./год",
})
input_table.index = station_specs["name"].to_numpy()
input_table.index.name = "Тип станции"
display(input_table)

,"a, МВт","b, МВт","c, ГВт·ч/год","d, млн долл.","f, млн долл./год"
Тип станции,,,,,
ГЭС,50.000000,100,400.000000,300.000000,6.000000
ТЭЦ,160.000000,200,"1,000.000000",400.000000,10.000000
АЭС,900.000000,1000,"7,500.000000","6,000.000000",180.000000
ВЭС,0.000000,100,350.000000,120.000000,4.500000
СЭС,0.000000,100,180.000000,70.000000,1.500000


**Прямая задача.** Все четыре ресурсных неравенства записаны в той же форме, что и в математической модели. После решения восстанавливаем исходные количества станций.

In [3]:
primal = pyo.ConcreteModel()
primal.stations = pyo.Set(initialize=station_data.index.tolist(), ordered=True)
primal.y = pyo.Var(primal.stations, domain=pyo.NonNegativeReals)
primal.scale = pyo.Var(domain=pyo.NonNegativeReals)
primal.firm_bound = pyo.Constraint(
    expr=required_firm * primal.scale
    - sum(station_data.loc[station, "a"] * primal.y[station] for station in primal.stations) <= 0
)
primal.peak_bound = pyo.Constraint(
    expr=required_peak * primal.scale
    - sum(station_data.loc[station, "b"] * primal.y[station] for station in primal.stations) <= 0
)
primal.energy_bound = pyo.Constraint(
    expr=required_energy * primal.scale
    - sum(station_data.loc[station, "c"] * primal.y[station] for station in primal.stations) <= 0
)
primal.budget_bound = pyo.Constraint(
    expr=sum(station_data.loc[station, "d"] * primal.y[station] for station in primal.stations)
    - construction_budget * primal.scale <= 0
)
primal.normalization = pyo.Constraint(
    expr=sum(station_data.loc[station, "d"] * primal.y[station] for station in primal.stations) == 1
)
primal.efficiency = pyo.Objective(
    expr=sum(station_data.loc[station, "c"] * primal.y[station] for station in primal.stations),
    sense=pyo.maximize,
)
primal.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
primal_result = solve_model(primal)

scale_value = pyo.value(primal.scale)
assert scale_value > 0
station_quantities = pd.Series({
    station: pyo.value(primal.y[station]) / scale_value for station in primal.stations
})
station_totals = station_data.mul(station_quantities, axis=0)

production_table = pd.DataFrame({
    "Тип станции": station_specs["name"],
    "y": [pyo.value(primal.y[station]) for station in primal.stations],
    "Количество x": station_quantities,
    "Строительство, млн долл.": station_totals["d"],
    "Выработка, ГВт·ч/год": station_totals["c"],
    "Эксплуатация, млн долл./год": station_totals["f"],
}).set_index("Тип станции")
display(production_table)

total_firm = station_totals["a"].sum()
total_peak = station_totals["b"].sum()
total_energy = station_totals["c"].sum()
total_construction = station_totals["d"].sum()
total_operation = station_totals["f"].sum()
original_efficiency = total_energy / total_construction

resource_table = pd.DataFrame({
    "Показатель": ["Гарантированная мощность, МВт", "Пиковая мощность, МВт",
                   "Годовая выработка, ГВт·ч/год", "Строительство, млн долл."],
    "Фактически": [total_firm, total_peak, total_energy, total_construction],
    "Условие": ["не менее", "не менее", "не менее", "не более"],
    "Граница": [required_firm, required_peak, required_energy, construction_budget],
    "Запас": [total_firm - required_firm, total_peak - required_peak,
              total_energy - required_energy, construction_budget - total_construction],
}).set_index("Показатель")
display(resource_table)
print(f"t = {scale_value:.10f}")
print(f"Годовые эксплуатационные расходы: {total_operation:.2f} млн долл./год")

,y,Количество x,"Строительство, млн долл.","Выработка, ГВт·ч/год","Эксплуатация, млн долл./год"
Тип станции,,,,,
ГЭС,0.000000,0.000000,0.000000,0.000000,0.000000
ТЭЦ,0.001538,2.000000,800.000000,"2,000.000000",20.000000
АЭС,0.000000,0.000000,0.000000,0.000000,0.000000
ВЭС,0.002308,3.000000,360.000000,"1,050.000000",13.500000
СЭС,0.001538,2.000000,140.000000,360.000000,3.000000


,Фактически,Условие,Граница,Запас
Показатель,,,,
"Гарантированная мощность, МВт",320.000000,не менее,320.000000,0.000000
"Пиковая мощность, МВт",900.000000,не менее,900.000000,0.000000
"Годовая выработка, ГВт·ч/год","3,410.000000",не менее,"3,200.000000",210.000000
"Строительство, млн долл.","1,300.000000",не более,"1,300.000000",0.000000


t = 0.0007692308
Годовые эксплуатационные расходы: 36.50 млн долл./год


**Двойственная задача.** Модель строится отдельно. Затем её решение сопоставляется с оценками ограничений, полученными решателем для прямой ЛП.

In [4]:
dual = pyo.ConcreteModel()
dual.stations = pyo.Set(initialize=station_data.index.tolist(), ordered=True)
dual.firm_price = pyo.Var(domain=pyo.NonNegativeReals)
dual.peak_price = pyo.Var(domain=pyo.NonNegativeReals)
dual.energy_price = pyo.Var(domain=pyo.NonNegativeReals)
dual.budget_price = pyo.Var(domain=pyo.NonNegativeReals)
dual.normalization_price = pyo.Var(domain=pyo.Reals)

def station_dual_rule(model, station):
    return (
        (model.normalization_price + model.budget_price) * station_data.loc[station, "d"]
        - model.firm_price * station_data.loc[station, "a"]
        - model.peak_price * station_data.loc[station, "b"]
        - model.energy_price * station_data.loc[station, "c"]
        >= station_data.loc[station, "c"]
    )

dual.station_bound = pyo.Constraint(dual.stations, rule=station_dual_rule)
dual.scale_bound = pyo.Constraint(
    expr=required_firm * dual.firm_price + required_peak * dual.peak_price
    + required_energy * dual.energy_price - construction_budget * dual.budget_price >= 0
)
dual.bound = pyo.Objective(expr=dual.normalization_price, sense=pyo.minimize)
dual_result = solve_model(dual)

dual_variables = [dual.firm_price, dual.peak_price, dual.energy_price,
                  dual.budget_price, dual.normalization_price]
primal_constraints = [primal.firm_bound, primal.peak_bound, primal.energy_bound,
                      primal.budget_bound, primal.normalization]
dual_values = np.array([pyo.value(variable) for variable in dual_variables])
imported_duals = np.array([primal.dual[constraint] for constraint in primal_constraints])
dual_table = pd.DataFrame({
    "Переменная": ["u", "v", "w", "z", "λ"],
    "Ограничение прямой ЛП": ["Гарантированная мощность", "Пиковая мощность",
                              "Годовая выработка", "Бюджет", "Нормировка"],
    "Явная двойственная модель": dual_values,
    "Оценка решателя прямой ЛП": imported_duals,
}).set_index("Переменная")
display(dual_table)

parameter_effects = scale_value * dual_values[:4] * np.array([-1, -1, -1, 1])
parameter_effects[np.abs(parameter_effects) < 1e-12] = 0.0
sensitivity_table = pd.DataFrame({
    "Исходный параметр": ["A, МВт", "B, МВт", "C, ГВт·ч/год", "D, млн долл."],
    "Предельное изменение эффективности": parameter_effects,
}).set_index("Исходный параметр")
display(sensitivity_table)

,Ограничение прямой ЛП,Явная двойственная модель,Оценка решателя прямой ЛП
Переменная,,,
u,Гарантированная мощность,1.525000,1.525000
v,Пиковая мощность,0.580000,0.580000
w,Годовая выработка,0.000000,0.000000
z,Бюджет,0.776923,0.776923
λ,Нормировка,2.623077,2.623077


,Предельное изменение эффективности
Исходный параметр,
"A, МВт",-0.001173
"B, МВт",-0.000446
"C, ГВт·ч/год",0.000000
"D, млн долл.",0.000598


**Проверка оптимальности.** Сравним дробную цель восстановленного плана, цель прямой ЛП и цель двойственной ЛП. Дополнительно проверим допустимость и условия дополняющей нежёсткости.

In [5]:
primal_value = pyo.value(primal.efficiency)
dual_value = pyo.value(dual.bound)
duality_gap = abs(dual_value - primal_value)
primal_slacks = np.array([-pyo.value(constraint.body) for constraint in primal_constraints[:4]])
station_dual_slacks = np.array([
    pyo.value(dual.station_bound[station].body - dual.station_bound[station].lower)
    for station in dual.stations
])
scale_dual_slack = pyo.value(dual.scale_bound.body)
y_values = np.array([pyo.value(primal.y[station]) for station in primal.stations])
complementarity_error = max(
    np.max(np.abs(dual_values[:4] * primal_slacks)),
    np.max(np.abs(y_values * station_dual_slacks)),
    abs(scale_value * scale_dual_slack),
)

assert np.all(primal_slacks >= -1e-8)
assert np.all(station_dual_slacks >= -1e-8)
assert scale_dual_slack >= -1e-8
assert np.isclose(pyo.value(primal.normalization.body), 1, atol=1e-8)
assert np.all(station_quantities.to_numpy() >= -1e-8)
assert np.allclose([primal_value, dual_value], original_efficiency, rtol=0, atol=1e-8)
assert complementarity_error < 1e-8

objective_table = pd.DataFrame({
    "Показатель": ["Исходная дробная цель η(x*)", "Прямая ЛП P*", "Двойственная ЛП Q*"],
    "ГВт·ч/год на 1 млн долл.": [original_efficiency, primal_value, dual_value],
}).set_index("Показатель")
with pd.option_context("display.float_format", "{:.10f}".format):
    display(objective_table)
certificate_table = pd.DataFrame({
    "Проверка": ["Разрыв двойственности |Q* − P*|", "Ошибка дополняющей нежёсткости"],
    "Значение": [duality_gap, complementarity_error],
}).set_index("Проверка")
with pd.option_context("display.float_format", "{:.3e}".format):
    display(certificate_table)
print("Допустимость и сильная двойственность подтверждены с точностью 1e-8.")

,ГВт·ч/год на 1 млн долл.
Показатель,
Исходная дробная цель η(x*),2.6230769231
Прямая ЛП P*,2.6230769231
Двойственная ЛП Q*,2.6230769231


,Значение
Проверка,
Разрыв двойственности |Q* − P*|,0.000e+00
Ошибка дополняющей нежёсткости,3.061e-16


Допустимость и сильная двойственность подтверждены с точностью 1e-8.


## Ответ

Для заданного учебного примера оптимальные количества станций:

| Тип | ГЭС | ТЭЦ | АЭС | ВЭС | СЭС |
|:---|---:|---:|---:|---:|---:|
| $x_j^*$ | 0 | 2 | 0 | 3 | 2 |

Максимальная эффективность составляет

$$\eta^*=\frac{3410}{1300}=\frac{341}{130}
\approx2{,}6230769231\ \frac{\text{ГВт·ч/год}}{\text{млн долл.}}.$$

Значения прямой и двойственной целевых функций совпадают:

$$\boxed{P^*=Q^*=\frac{341}{130}\approx2{,}6230769231.}$$

Сильная двойственность подтверждена: численный разрыв между оптимальными значениями не превышает $10^{-8}$.